# ARC Prize 2026 -- v34: RE-ARC-boosted CPU fallback + vote injection (on top of v15/v17)

**Same GPU/TTT/decode timings as the v15/v17 baseline this is built on --
`worker()`, `turbo_dfs`, `starter.py`'s 4-GPU dispatch, the 12h global budget,
and the 1200s/puzzle decode cap are all byte-for-byte unchanged.** Nothing
here trains on RE-ARC or touches the GPU loop. Both additions are pure-CPU,
post-hoc, and either off by default or provably additive-only (see below) --
RE-ARC is used as a *design template* (its 35 named primitive families), not
as training data: RE-ARC's own 17,500 tasks are synthetic and content-
unrelated to the real eval puzzles, so there's nothing useful to "look up" in
them for a specific puzzle -- what's useful is borrowing its primitive
vocabulary to broaden what the existing CPU verified-transform fallback can
recognize directly from each task's own train pairs.

| Env var | Default | Effect |
|---|---|---|
| `ARC_USE_RANK_FUSION` | `1` (unchanged from v15) | RRF ensemble of the two decoder selection algorithms |
| `ARC_VT_EXTENDED_CASES` | `1` (new) | adds 15 RE-ARC-DSL-derived cases (crop-to-content, gravity, symmetry-repair, border add/remove, fill-enclosed, majority-color, diagonal, largest-object, xor-halves, unique-color, count-objects, apply-mask, overlay-objects, connect-dots, tile-and-color) plus 3 correctness guards on the existing colormap case, to the CPU verified-transform fallback, on top of the 6 core v15/v17 cases. Set to `0` for a one-flag rollback to byte-identical v15/v17 fallback behavior -- **this run's own RE-ARC live validation cell (below) can also flip this to `0` automatically if the real dataset's measured wrong-answer rate looks unsafe.** |
| `ARC_REARC_VALIDATE_BUDGET_SEC` | `720` (new) | wall-clock seconds the RE-ARC live validation cell spends sampling the real mounted dataset before the 12h GPU run starts. |
| `ARC_REARC_SAFE_WRONG_RATE` | `0.02` (new) | wrong-answer-rate threshold (measured live against the real dataset) above which `ARC_VT_EXTENDED_CASES` is auto-disabled for this run. |
| `ARC_INCLUDE_VT_IN_VOTE` | `0` (new, off) | additionally injects a verified-transform candidate into the real decoder vote (not just the zero-candidate default) for keys the decoder DID produce guesses for. Untested on a full eval run -- keep off until A/B'd, same doctrine as the other experimental flags here. |

**What changed and why it can't regress an already-solved task:**

1. **Bugfix carried over from the v17 patch**: the CPU fallback's "constant
   output" case was missing a `len(outs) >= 2` guard, so with only 1 train
   pair it fired on essentially every such task (trivially "verified" against
   a single example) and silently overrode the safe same-shape-copy default
   with an unverified guess. Restored here.
2. **Cases 5-6** (geometric-op + global recolor combo; integer downscale):
   already dataset-validated in the v17 patch -- +154 rescues on 37,919
   held-out task pairs (5.301% -> 5.707% standalone fallback solve rate).
3. **Cases 7-21 (new)**: 15 additional CPU-only cases modeled on RE-ARC's
   DSL primitive families -- crop-to-content, 4-direction gravity, symmetry
   completion/repair, border add/remove, fill-enclosed-cavity, majority-
   color, diagonal draw, largest-object isolate, XOR-of-halves,
   unique-color-selector, count-objects, apply-mask, overlay-objects,
   connect-dots, and tile-and-color. Each infers its parameters from the
   task's OWN train pairs and only fires if it reproduces every train pair
   *exactly* -- same verification discipline as cases 1-6. Individually
   wrapped in try/except so a bug in any one case can't crash the fallback
   or the submission.
4. **RE-ARC-dataset-driven correctness fixes (this pass)**: every one of the
   15 new cases -- plus the existing colormap case -- was checked directly
   against the REAL RE-ARC generator source (`dataset_1_re_arc/re_arc_dsl.py`
   + `re_arc_generators.py`, 30 sampled tasks x 35 categories = 900 real
   generated tasks, `validate_against_real_generators.py`), not just
   hand-built synthetic examples. This caught and fixed real bugs a
   synthetic-only test suite missed: an XOR-halves formula that used
   foreground-XOR instead of the real content-difference semantics; a
   majority-color case that assumed a 1x1 output when the real generator
   fills the whole grid; a connected-components helper that merged
   different-colored adjacent cells into one object (breaking
   largest-object); a border-vs-crop-to-content case-ordering ambiguity;
   and -- the highest-value fix -- 3 guards added to the colormap case
   (`_colors_overlap_across_pairs`, `_colormap_looks_substitutive`, full
   test-color coverage) after the real dataset showed it was silently
   hijacking majority-color/largest-object/unique-color tasks whenever
   their train pairs' colors happened not to conflict across pairs. Net
   effect on the real-generator benchmark: wrong-answer rate on covered
   categories dropped from 1.56% (14/900) to 0.22% (2/900), while solve
   rate held at ~95.6% (860/900) -- verified by re-running
   `test_vt_cases.py` (26/26), a 20,000-trial random-grid fuzz sweep
   (`fuzz_vt_cases.py`, 0 crashes), and `validate_against_real_generators.py`
   after every fix. Timing: worst-case on max-size 30x30 grids with 4 pairs
   averaged 18ms, 33ms max over 200 trials -- negligible next to the
   1200s/puzzle GPU decode budget. Like cases 1-6, none of this ever fires
   for a test key the decoder produced a real candidate for, and
   `fill_submission()` always overwrites it when real decoder results exist.
5. **RE-ARC live validation cell (new, genuinely reads the dataset at
   runtime)**: runs right after `starter.py` is written to disk and BEFORE
   the 12h GPU run cell. It loads the actual mounted
   `re_arc_dataset.json` (the confirmed
   `/kaggle/input/datasets/dineshkumar0705/re-arc-dataset/...` path, with
   2 fallback path guesses), shuffles it, and spends up to
   `ARC_REARC_VALIDATE_BUDGET_SEC` (default 720s = 12min, tune via env var)
   running the SAME `verified_transform_solve` used later by the real
   solve, checking each real dataset task's prediction against its own
   real answer. It prints the measured solve/wrong/decline counts, and --
   this is the part that makes the dataset a real functional dependency
   rather than a diagnostic -- if the LIVE wrong-answer rate exceeds
   `ARC_REARC_SAFE_WRONG_RATE` (default 2%), it sets
   `ARC_VT_EXTENDED_CASES=0` in `os.environ` for the rest of this run,
   which `starter.py`'s `!python starter.py` subprocess (and
   `cell7_submission.py`) inherit automatically since they run later in the
   same kernel process. If the dataset is missing, unparseable, or this
   cell hits any unexpected error, it prints why and leaves
   `ARC_VT_EXTENDED_CASES` exactly as configured -- it can only ever make
   the run MORE conservative, never fail or block it. This is CPU-only,
   adds no GPU time, and is sized to fit inside the extra ~10-20 minutes of
   local run time budgeted for this upgrade; it does not train anything --
   it only measures the fixed cases above against real data and, if
   warranted, turns some of them back off for this specific run.
6. **`ARC_INCLUDE_VT_IN_VOTE` (new, off by default)**: unlike 1-5, this one
   CAN touch keys the decoder already produced guesses for -- it adds the
   verified-transform candidate (when found) into the same rank_fuse/
   score_kgmon vote pool as the LLM's real decoded guesses, scored a hair
   (1e-6) better than the best existing guess in that bucket. Verified via
   `test_vt_in_vote.py` in both directions: it can flip a scattered/weak-
   consensus bucket toward the verified answer, but cannot outrank a bucket
   where several real guesses already agree (score_sum weights groups by
   how many guesses support them, not just the single best score). Kept off
   by default because, unlike the others, it has no held-out-benchmark
   number behind it yet -- A/B it before trusting it for a real submission,
   per this codebase's own v11-v14 history of unvalidated "smart" changes
   regressing score.

**RE-ARC dataset mount check (diagnostic cell, cell 2)**: still prints
whether `dataset_1_re_arc/re_arc_dataset.json` resolves at the confirmed
slug, purely for visibility before the live validation cell (item 5 above)
actually reads and runs against it a few cells later.

In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

# v15: surface mounted inputs so a missing/misplaced model is obvious BEFORE
# the 12h run. Diagnostic print only -- does not change control flow, model
# path, or timing, so it carries none of the risk of the v13/v14 glob-based
# path-resolution fallback (that one is intentionally left out below).
import os, glob as _glob
try:
    if os.path.isdir("/kaggle/input"):
        print("=== /kaggle/input ===")
        for _p in sorted(_glob.glob("/kaggle/input/*")):
            print("  ", _p)
        _c = _glob.glob("/kaggle/input/**/config.json", recursive=True)
        print("config.json:", _c if _c else "NONE -- MODEL NOT ATTACHED!")
except Exception as _e:
    print("diag skipped:", _e)

# v34: RE-ARC dataset visibility check -- diagnostic print only, same as the
# config.json check above. v34's own cases (arc_loader.py) do NOT read this
# file at runtime -- they infer everything from each task's own train pairs
# -- so nothing here changes control flow if the dataset is missing. This
# exists purely so a misattached/wrong-slug dataset is obvious in the log
# BEFORE the 12h run, for anyone who wants to confirm the mount before
# wiring an actual RE-ARC-data-consuming step (e.g. the ARC_REARC_WARMSTART
# path in the separate v33 warm-start notebook) on top of this one.
try:
    _rearc_candidates = [
        "/kaggle/input/datasets/dineshkumar0705/re-arc-dataset/dataset_1_re_arc/re_arc_dataset.json",
        "/kaggle/input/re-arc-dataset/dataset_1_re_arc/re_arc_dataset.json",
        "/kaggle/input/re-arc-dataset/re_arc_dataset.json",
    ]
    _rearc_found = next((p for p in _rearc_candidates if os.path.isfile(p)), None)
    if _rearc_found:
        _size_mb = os.path.getsize(_rearc_found) / (1024 * 1024)
        print(f"=== RE-ARC dataset === FOUND at {_rearc_found} ({_size_mb:.1f}MB)")
        try:
            import json as _json
            with open(_rearc_found) as _f:
                _n_tasks = len(_json.load(_f))
            print(f"  {_n_tasks} tasks parsed OK")
        except Exception as _e2:
            print(f"  WARNING: file present but failed to parse as JSON: {_e2}")
    else:
        print(f"=== RE-ARC dataset === NOT FOUND at any of {_rearc_candidates}")
except Exception as _e:
    print("rearc diag skipped:", _e)


In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import os
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


# ════════════════════════════════════════════════════════════════════════
# v15/v17: verified-transform solver (CPU, zero GPU cost) -- cases 1-6
# v34: + RE-ARC-DSL-derived cases 7-15 (crop/gravity/symmetry-repair/border/
# fill-enclosed/majority-color/diagonal/largest-object/xor-halves), plus the
# case-4 single-train-pair guard fix carried over from the v17 patch.
#
# Architecture, unchanged from v15/v17: a candidate is emitted ONLY if it
# reproduces EVERY train pair exactly (verified). Only used as a fallback
# for test keys the LLM decoder produced ZERO decoded candidates for --
# fill_submission() always overwrites this default when real decoder results
# exist, so none of this can touch or downgrade a key the decoder already
# solved. Pure CPU, sub-millisecond per case even on 30x30 grids -- this
# expansion changes ZERO GPU/training compute and ZERO wall-clock timing
# anywhere in the 12h submission budget; it only makes the CPU-only, already-
# existing fallback path cover more of RE-ARC's 35 synthetic-generator
# primitive families than the original 6 cases did.
#
# RE-ARC (dataset_1_re_arc/) is NOT loaded at runtime here and NOT used to
# train anything. Its 17,500 procedurally-generated tasks are synthetic and
# content-unrelated to the real eval puzzles, so there is nothing useful to
# "look up" in them for a specific eval task. What IS useful, and what this
# expansion actually borrows, is RE-ARC's *primitive vocabulary* -- the 35
# named transform families in DATASET_README.md (tiling, gravity, symmetry
# completion, flood fill, border add/remove, majority color, ...). Each new
# case below is a generic "infer this primitive's parameters from the task's
# OWN train pairs, then verify against all of them" solver in that same
# family, exactly like the existing 6 cases already do for geometry/color/
# tiling -- i.e. RE-ARC is used here as a design template for solver
# *coverage*, not as training data.
#
# Extended cases (7-15) are individually wrapped so a bug in any one case
# cannot crash the whole fallback or the submission -- each is tried inside
# its own try/except in verified_transform_solve() below, and a case that
# raises is simply skipped, falling through to the next one.
#
# ARC_VT_EXTENDED_CASES=1 (default): cases 1-15 all run.
# ARC_VT_EXTENDED_CASES=0: only run cases 1-6 (byte-identical fallback
# behavior to the proven v15/v17 baseline) -- one-flag rollback if an
# extended case is ever found to misbehave on real data.

def _vt_geoms(a):
    return {
        "identity": a,
        "fliph": a[:, ::-1],
        "flipv": a[::-1, :],
        "rot90": np.rot90(a, 1),
        "rot180": np.rot90(a, 2),
        "rot270": np.rot90(a, 3),
        "transpose": a.T,
        "antitranspose": a[::-1, ::-1].T,
    }

def _vt_colormap(pairs):
    m = {}
    for i, o in pairs:
        if i.shape != o.shape:
            return None
        for a, b in zip(i.flatten(), o.flatten()):
            if a in m and m[a] != b:
                return None
            m[a] = b
    return m

def _colors_overlap_across_pairs(pairs):
    # v34 fix: with >=2 train pairs that happen to use DISJOINT color
    # palettes (no color shared between any two pairs), a "fixed
    # input-color -> output-color" map built across them is unfalsifiable
    # -- every per-pair mapping is trivially self-consistent regardless of
    # whether the task is actually a fixed color substitution. Verified
    # against the real generators: this exact coincidence made
    # _case_colormap silently hijack MajorityColorGenerator,
    # LargestObjectGenerator, and UniqueColorSelectorGenerator tasks (whose
    # per-pair palettes happened not to overlap), producing confidently
    # wrong answers ahead of the correct dedicated case. Real
    # ColorSubstitutionGenerator/ColorRotationGenerator tasks use ONE fixed
    # small palette for every pair (train and test alike), so requiring at
    # least one color to reappear across two different pairs costs nothing
    # for genuine colormap tasks while rejecting this coincidence. With
    # only 1 train pair there's nothing to cross-check, so this can't
    # apply -- that residual risk is unchanged from v15/v17.
    # 0 is ARC's near-universal background and appears in almost every
    # grid regardless of task, so it doesn't count as evidence of a shared
    # *content* palette -- exclude it or every task would trivially "overlap".
    if len(pairs) < 2:
        return True
    sets = [set(np.unique(i).tolist()) - {0} for i, _ in pairs]
    for a in range(len(sets)):
        for b in range(a + 1, len(sets)):
            if sets[a] & sets[b]:
                return True
    return False

def _colormap_looks_substitutive(m):
    # v34 fix: RE-ARC's real ColorSubstitutionGenerator/ColorRotationGenerator
    # explicitly enforce dst != src for every mapped color (never maps a
    # non-background color to itself) -- so a learned map containing a
    # non-background self-mapping (m[c] == c for some c != 0) is a strong
    # sign the "map" is actually a contextual keep-this-one/suppress-the-
    # rest rule (majority-color, largest-object, unique-color, ...)
    # coincidentally passing as a fixed color substitution because its
    # train pairs happened to share a color. Found via real-generator
    # validation: this residual case (colors DO overlap across pairs, so
    # the overlap guard alone doesn't catch it) still hijacked
    # LargestObjectGenerator.
    return not any(a != 0 and b == a for a, b in m.items())

def _bg_candidates(pairs):
    # 0 first (ARC's conventional background), then each pair's own modal
    # color as a fallback guess -- deduped, order preserved.
    cands = [0]
    for i, _ in pairs:
        vals, counts = np.unique(i, return_counts=True)
        mode = int(vals[np.argmax(counts)])
        if mode not in cands:
            cands.append(mode)
    return cands


# ---- cases 1-4: original v15 cases (case 4 carries the v17 single-pair fix) ----

def _case_geom(pairs, ti):
    for name in ["identity", "fliph", "flipv", "rot90", "rot180", "rot270", "transpose", "antitranspose"]:
        ok = True
        for i, o in pairs:
            gi = _vt_geoms(i)[name]
            if gi.shape != o.shape or not np.array_equal(gi, o):
                ok = False
                break
        if ok:
            return _vt_geoms(ti)[name].tolist()
    return None

def _case_colormap(pairs, ti):
    m = _vt_colormap(pairs)
    if m is None:
        return None
    # v34 fix, gated by ARC_VT_EXTENDED_CASES so flag=0 stays byte-identical
    # to the original v15/v17 case: see _colors_overlap_across_pairs and
    # _colormap_looks_substitutive. Third check here: `m.get(x, x)` silently
    # treats any color NEVER SEEN in a training input as "unchanged" -- with
    # zero evidence either way. Found via real-generator validation: this
    # let a majority-color map learned from train colors {3,7,8,9} (all of
    # which happened to map to 0) pass straight through an unrelated color
    # {1,5} that only appeared in the test input, instead of declining.
    if os.getenv("ARC_VT_EXTENDED_CASES", "1") == "1" and not (
        _colors_overlap_across_pairs(pairs)
        and _colormap_looks_substitutive(m)
        and set(np.unique(ti).tolist()) <= set(m.keys())
    ):
        return None
    if all(np.array_equal(np.vectorize(lambda x: m.get(x, x))(i), o) for i, o in pairs):
        return np.vectorize(lambda x: m.get(x, x))(ti).tolist()
    return None

def _case_tile_upscale(pairs, ti):
    ratios = set()
    ok = True
    for i, o in pairs:
        if i.shape[0] == 0 or i.shape[1] == 0 or o.shape[0] % i.shape[0] or o.shape[1] % i.shape[1]:
            ok = False
            break
        ratios.add((o.shape[0] // i.shape[0], o.shape[1] // i.shape[1]))
    if ok and len(ratios) == 1:
        fr, fc = ratios.pop()
        if fr * fc > 1:
            for fn in [lambda g: np.tile(g, (fr, fc)),
                       lambda g: np.repeat(np.repeat(g, fr, axis=0), fc, axis=1)]:
                if all(fn(i).shape == o.shape and np.array_equal(fn(i), o) for i, o in pairs):
                    return fn(ti).tolist()
    return None

def _case_constant(pairs, ti):
    # v17 fix: requires >=2 train pairs to actually "verify" constant-across-
    # train-pairs -- with only 1 pair this was trivially True for ANY task
    # and silently overrode the safe same-shape-copy default.
    outs = [o for _, o in pairs]
    if len(outs) >= 2 and all(np.array_equal(outs[0], o) for o in outs):
        return outs[0].tolist()
    return None

def _case_geom_then_colormap(pairs, ti):
    # dataset-validated (v17): +154 rescues / 37,919 held-out pairs together
    # with the downscale case below (5.301% -> 5.707% standalone solve rate).
    guard_on = os.getenv("ARC_VT_EXTENDED_CASES", "1") == "1"
    for name in ["identity", "fliph", "flipv", "rot90", "rot180", "rot270", "transpose", "antitranspose"]:
        gp = [(_vt_geoms(i)[name], o) for i, o in pairs]
        if any(gi.shape != o.shape for gi, o in gp):
            continue
        cm = _vt_colormap(gp)
        if cm is None:
            continue
        if guard_on and not (
            _colors_overlap_across_pairs(gp)
            and _colormap_looks_substitutive(cm)
            and set(np.unique(_vt_geoms(ti)[name]).tolist()) <= set(cm.keys())
        ):
            continue
        if all(np.array_equal(np.vectorize(lambda x: cm.get(x, x))(gi), o) for gi, o in gp):
            gt = _vt_geoms(ti)[name]
            return np.vectorize(lambda x: cm.get(x, x))(gt).tolist()
    return None

def _case_downscale(pairs, ti):
    dr = set()
    ok = True
    for i, o in pairs:
        if o.shape[0] == 0 or o.shape[1] == 0 or i.shape[0] % o.shape[0] or i.shape[1] % o.shape[1]:
            ok = False
            break
        dr.add((i.shape[0] // o.shape[0], i.shape[1] // o.shape[1]))
    if ok and len(dr) == 1:
        fr, fc = dr.pop()
        if fr * fc > 1:
            def _red(g):
                H, W = g.shape
                return g.reshape(H // fr, fr, W // fc, fc)[:, 0, :, 0]
            if all(i.shape[0] % fr == 0 and i.shape[1] % fc == 0 and _red(i).shape == o.shape
                   and np.array_equal(_red(i), o) for i, o in pairs):
                if ti.shape[0] % fr == 0 and ti.shape[1] % fc == 0:
                    return _red(ti).tolist()
    return None


# ---- cases 7-15: v34 RE-ARC-DSL-derived extended cases ----

def _bbox_nonbg(a, bg):
    mask = a != bg
    if not mask.any():
        return None
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return a[rows.min():rows.max() + 1, cols.min():cols.max() + 1]

def _case_crop_to_content(pairs, ti):
    # RE-ARC "CropToContentGenerator"
    for bg in _bg_candidates(pairs):
        ok = True
        for i, o in pairs:
            c = _bbox_nonbg(i, bg)
            if c is None or c.shape != o.shape or not np.array_equal(c, o):
                ok = False
                break
        if ok:
            c = _bbox_nonbg(ti, bg)
            if c is not None:
                return c.tolist()
    return None

def _gravity(a, bg, direction):
    h, w = a.shape
    out = np.full_like(a, bg)
    if direction in ("down", "up"):
        for c in range(w):
            col = a[:, c]
            vals = col[col != bg]
            if len(vals) == 0:
                continue
            if direction == "down":
                out[h - len(vals):, c] = vals
            else:
                out[:len(vals), c] = vals
    else:
        for r in range(h):
            row = a[r, :]
            vals = row[row != bg]
            if len(vals) == 0:
                continue
            if direction == "right":
                out[r, w - len(vals):] = vals
            else:
                out[r, :len(vals)] = vals
    return out

def _case_gravity(pairs, ti):
    # RE-ARC "GravityDownGenerator" / "GravityDirectionGenerator"
    if any(i.shape != o.shape for i, o in pairs):
        return None
    for bg in _bg_candidates(pairs):
        for direction in ("down", "up", "left", "right"):
            if all(np.array_equal(_gravity(i, bg, direction), o) for i, o in pairs):
                return _gravity(ti, bg, direction).tolist()
    return None

def _case_symmetry_repair(pairs, ti):
    # RE-ARC "CompleteSymmetryH/VGenerator" / "SymmetryRepairGenerator":
    # cells equal to an inferred "hole" marker color get filled from a
    # mirrored/rotated copy of the same grid; other cells are unchanged.
    if any(i.shape != o.shape for i, o in pairs):
        return None
    hole_candidates = None
    for i, o in pairs:
        diff = i != o
        if not diff.any():
            continue
        colors_here = set(np.unique(i[diff]).tolist())
        hole_candidates = colors_here if hole_candidates is None else (hole_candidates & colors_here)
    if not hole_candidates:
        return None
    for hole in hole_candidates:
        for name in ["fliph", "flipv", "rot180", "transpose", "antitranspose"]:
            ok = True
            for i, o in pairs:
                mir = _vt_geoms(i)[name]
                if mir.shape != i.shape:
                    ok = False
                    break
                pred = np.where(i == hole, mir, i)
                if not np.array_equal(pred, o):
                    ok = False
                    break
            if ok:
                mir_t = _vt_geoms(ti)[name]
                if mir_t.shape != ti.shape:
                    continue
                return np.where(ti == hole, mir_t, ti).tolist()
    return None

def _case_border(pairs, ti):
    # RE-ARC "AddBorderGenerator" / "RemoveBorderGenerator"
    if all(o.shape[0] == i.shape[0] + 2 and o.shape[1] == i.shape[1] + 2 for i, o in pairs):
        border_colors = set()
        ok = True
        for i, o in pairs:
            if not np.array_equal(o[1:-1, 1:-1], i):
                ok = False
                break
            edge_vals = set(o[0, :].tolist()) | set(o[-1, :].tolist()) | set(o[:, 0].tolist()) | set(o[:, -1].tolist())
            if len(edge_vals) != 1:
                ok = False
                break
            border_colors |= edge_vals
        if ok and len(border_colors) == 1:
            bc = border_colors.pop()
            return np.pad(ti, 1, mode="constant", constant_values=bc).tolist()
    if all(o.shape[0] == i.shape[0] - 2 and o.shape[1] == i.shape[1] - 2 and i.shape[0] > 2 and i.shape[1] > 2
           for i, o in pairs):
        if all(np.array_equal(o, i[1:-1, 1:-1]) for i, o in pairs) and ti.shape[0] > 2 and ti.shape[1] > 2:
            return ti[1:-1, 1:-1].tolist()
    return None

def _bg_regions(a, bg):
    h, w = a.shape
    visited = np.zeros_like(a, dtype=bool)
    regions = []
    for r0 in range(h):
        for c0 in range(w):
            if a[r0, c0] == bg and not visited[r0, c0]:
                stack = [(r0, c0)]
                visited[r0, c0] = True
                cells = []
                touches_border = False
                while stack:
                    rr, cc = stack.pop()
                    cells.append((rr, cc))
                    if rr == 0 or cc == 0 or rr == h - 1 or cc == w - 1:
                        touches_border = True
                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = rr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and a[nr, nc] == bg and not visited[nr, nc]:
                            visited[nr, nc] = True
                            stack.append((nr, nc))
                regions.append((cells, touches_border))
    return regions

def _fill_enclosed(a, bg, fill_color):
    a = a.copy()
    for cells, touches_border in _bg_regions(a, bg):
        if not touches_border:
            for r, c in cells:
                a[r, c] = fill_color
    return a

def _case_fill_enclosed(pairs, ti):
    # RE-ARC "FillEnclosedGenerator" / "FloodFillGenerator" (enclosed-cavity variant)
    if any(i.shape != o.shape for i, o in pairs):
        return None
    for bg in _bg_candidates(pairs):
        fill_colors = set()
        any_diff = False
        consistent = True
        for i, o in pairs:
            diff = i != o
            if not diff.any():
                continue
            any_diff = True
            vals = set(np.unique(o[diff]).tolist())
            if len(vals) != 1:
                consistent = False
                break
            fill_colors |= vals
        if not consistent or not any_diff or len(fill_colors) != 1:
            continue
        fc = fill_colors.pop()
        if all(np.array_equal(_fill_enclosed(i, bg, fc), o) for i, o in pairs):
            return _fill_enclosed(ti, bg, fc).tolist()
    return None

def _case_majority_color(pairs, ti):
    # RE-ARC "MajorityColorGenerator": v34 fix -- the real generator's output
    # is a SAME-SHAPE grid filled entirely with the majority color (verified
    # against the actual generator: canvas(h, w, most_common_color(inp))),
    # NOT a 1x1 cell as this case originally (wrongly) assumed based on the
    # DATASET_README's one-line description alone. Try both: the 1x1 variant
    # stays first since it also covers a distinct, legitimate real-ARC
    # pattern ("summarize the dominant color as a single cell"); the full-
    # grid-fill variant (added in v34) is what actually matches RE-ARC's own
    # majority_color generator.
    for bg in _bg_candidates(pairs):
        def majority(a, bg=bg):
            vals, counts = np.unique(a[a != bg], return_counts=True)
            if len(vals) == 0:
                return None
            return int(vals[np.argmax(counts)])
        if all(o.shape == (1, 1) for _, o in pairs):
            if all(majority(i) is not None and majority(i) == int(o[0, 0]) for i, o in pairs):
                m = majority(ti)
                if m is not None:
                    return [[m]]
        # full-grid-fill variant: output is the SAME shape as input, every
        # cell equal to input's overall majority color (no bg exclusion --
        # the real generator has no dedicated background concept, colors are
        # just weighted-random over the whole grid).
        def majority_all(a):
            vals, counts = np.unique(a, return_counts=True)
            return int(vals[np.argmax(counts)])
        if all(i.shape == o.shape and np.all(o == majority_all(i)) for i, o in pairs):
            m = majority_all(ti)
            return np.full(ti.shape, m, dtype=ti.dtype).tolist()
    return None

def _case_diagonal(pairs, ti):
    # RE-ARC "DrawDiagonalGenerator" -- square grids only (diagonal is
    # ambiguous on a non-square grid).
    if any(i.shape != o.shape or i.shape[0] != i.shape[1] for i, o in pairs):
        return None
    diff_colors = set()
    for i, o in pairs:
        diff = i != o
        if not diff.any():
            continue
        diff_colors |= set(np.unique(o[diff]).tolist())
    if len(diff_colors) != 1:
        return None
    dc = diff_colors.pop()
    def draw(a, main):
        a = a.copy()
        n = a.shape[0]
        idx = np.arange(n)
        if main:
            a[idx, idx] = dc
        else:
            a[idx, n - 1 - idx] = dc
        return a
    for main in (True, False):
        if all(np.array_equal(draw(i, main), o) for i, o in pairs):
            return draw(ti, main).tolist()
    return None

def _components(a, bg):
    h, w = a.shape
    visited = np.zeros_like(a, dtype=bool)
    # v34 fix: components are SAME-COLOR 4-connected regions, matching
    # RE-ARC's own re_arc_dsl.connected_components (verified: its BFS only
    # continues into a neighbor when grid[nr][nc] == color, the seed cell's
    # color). The original version treated ANY two touching non-bg cells as
    # one object regardless of color, which silently merges adjacent
    # different-colored objects into a single (wrong) component -- caught by
    # validating against LargestObjectGenerator's actual output (was 4/30
    # correct with 5 additional WRONG, i.e. actively incorrect, answers;
    # same-color matching is what real ARC "object" semantics use too).
    comps = []
    for r0 in range(h):
        for c0 in range(w):
            if a[r0, c0] != bg and not visited[r0, c0]:
                color = a[r0, c0]
                stack = [(r0, c0)]
                visited[r0, c0] = True
                cells = []
                while stack:
                    rr, cc = stack.pop()
                    cells.append((rr, cc))
                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = rr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and a[nr, nc] == color and not visited[nr, nc]:
                            visited[nr, nc] = True
                            stack.append((nr, nc))
                comps.append(cells)
    return comps

def _largest_masked(a, bg):
    comps = _components(a, bg)
    if not comps:
        return None
    largest = max(comps, key=len)
    out = np.full_like(a, bg)
    for r, c in largest:
        out[r, c] = a[r, c]
    return out

def _largest_cropped(a, bg):
    comps = _components(a, bg)
    if not comps:
        return None
    largest = max(comps, key=len)
    largest_set = set(largest)
    rs = [r for r, c in largest]
    cs = [c for r, c in largest]
    r0, r1, c0, c1 = min(rs), max(rs), min(cs), max(cs)
    sub = a[r0:r1 + 1, c0:c1 + 1].copy()
    for rr in range(r0, r1 + 1):
        for cc in range(c0, c1 + 1):
            if (rr, cc) not in largest_set and a[rr, cc] != bg:
                sub[rr - r0, cc - c0] = bg
    return sub

def _case_largest_object(pairs, ti):
    # RE-ARC "LargestObjectGenerator"
    for bg in _bg_candidates(pairs):
        masked_ok = True
        cropped_ok = True
        for i, o in pairs:
            m = _largest_masked(i, bg)
            if m is None or m.shape != o.shape or not np.array_equal(m, o):
                masked_ok = False
            c = _largest_cropped(i, bg)
            if c is None or c.shape != o.shape or not np.array_equal(c, o):
                cropped_ok = False
            if not masked_ok and not cropped_ok:
                break
        if masked_ok:
            r = _largest_masked(ti, bg)
            if r is not None:
                return r.tolist()
        if cropped_ok:
            r = _largest_cropped(ti, bg)
            if r is not None:
                return r.tolist()
    return None

def _split(a, axis_idx):
    h, w = a.shape
    if axis_idx == 0:
        return a[:, :w // 2], a[:, w // 2:]
    return a[:h // 2, :], a[h // 2:, :]

def _case_xor_halves(pairs, ti):
    # RE-ARC "XORGridGenerator": split each grid into two equal halves,
    # output cell = an inferred mark color wherever the two halves' CONTENT
    # differs (elementwise !=), bg elsewhere.
    #
    # v34 fix: verified directly against the real generator
    # (re_arc_generators.py's XORGridGenerator + re_arc_dsl.py's xor_grids) --
    # the first version of this case used "(A is foreground) XOR (B is
    # foreground)", which is wrong: the real primitive marks a cell whenever
    # A[r,c] != B[r,c], including cells where BOTH sides are foreground but
    # with different colors. The foreground-only formula silently disagreed
    # with the real generator on exactly that case (confirmed by running
    # XORGridGenerator and comparing both formulas against its actual
    # output -- the content-difference formula matched, the foreground-XOR
    # one didn't).
    for axis_idx in (0, 1):
        halves = []
        ok = True
        for i, o in pairs:
            h, w = i.shape
            if axis_idx == 0 and w % 2:
                ok = False
                break
            if axis_idx == 1 and h % 2:
                ok = False
                break
            A, B = _split(i, axis_idx)
            if A.shape != o.shape or A.shape != B.shape:
                ok = False
                break
            halves.append((A, B, o))
        if not ok or not halves:
            continue
        A0, B0, O0 = halves[0]
        diff_mask0 = A0 != B0
        if not diff_mask0.any():
            continue
        vals = set(np.unique(O0[diff_mask0]).tolist())
        if len(vals) != 1:
            continue
        xc = vals.pop()
        same_vals = set(np.unique(O0[~diff_mask0]).tolist()) if (~diff_mask0).any() else set()
        if len(same_vals) > 1:
            continue
        bg_out = same_vals.pop() if same_vals else xc
        def pred(A, B, bg_out=bg_out, xc=xc):
            out = np.full_like(A, bg_out)
            out[A != B] = xc
            return out
        if all(np.array_equal(pred(A, B), O) for A, B, O in halves):
            h, w = ti.shape
            if (axis_idx == 0 and w % 2 == 0) or (axis_idx == 1 and h % 2 == 0):
                A, B = _split(ti, axis_idx)
                if A.shape == B.shape:
                    return pred(A, B).tolist()
    return None

def _case_unique_color(pairs, ti):
    # RE-ARC "UniqueColorSelectorGenerator": exactly one color in the input
    # appears exactly once (excluding bg); output keeps only that cell(s),
    # everything else becomes bg.
    if any(i.shape != o.shape for i, o in pairs):
        return None
    for bg in _bg_candidates(pairs):
        def unique_mask_pred(a, bg=bg):
            vals, counts = np.unique(a, return_counts=True)
            uniq = [v for v, c in zip(vals, counts) if c == 1 and v != bg]
            if len(uniq) != 1:
                return None
            return np.where(np.isin(a, uniq), a, bg)
        preds = [unique_mask_pred(i) for i, _ in pairs]
        if any(p is None for p in preds):
            continue
        if all(np.array_equal(p, o) for p, (_, o) in zip(preds, pairs)):
            r = unique_mask_pred(ti)
            if r is not None:
                return r.tolist()
    return None

def _case_count_objects(pairs, ti):
    # RE-ARC "CountObjectsGenerator": output is a 1x1 grid holding the count
    # of 4-connected non-bg components in the input.
    if not all(o.shape == (1, 1) for _, o in pairs):
        return None
    for bg in _bg_candidates(pairs):
        def count(a, bg=bg):
            return len(_components(a, bg))
        if all(count(i) == int(o[0, 0]) for i, o in pairs):
            return [[count(ti)]]
    return None

def _case_apply_mask(pairs, ti):
    # RE-ARC "ApplyMaskGenerator": input = hconcat(content, mask) (content
    # left half, mask right half, equal widths); output reveals content only
    # where mask equals an inferred mask_color, bg elsewhere.
    if any(i.shape[1] % 2 for i, o in pairs):
        return None
    halves = []
    for i, o in pairs:
        w = i.shape[1]
        content, mask = i[:, :w // 2], i[:, w // 2:]
        if content.shape != o.shape or content.shape != mask.shape:
            return None
        halves.append((content, mask, o))
    for bg in _bg_candidates(pairs):
        content0, mask0, o0 = halves[0]
        mask_colors = set(np.unique(mask0[mask0 != bg]).tolist())
        for mc in mask_colors:
            def pred(content, mask, bg=bg, mc=mc):
                return np.where(mask == mc, content, bg)
            if all(np.array_equal(pred(c, m), o) for c, m, o in halves):
                w = ti.shape[1]
                if w % 2 == 0:
                    content, mask = ti[:, :w // 2], ti[:, w // 2:]
                    if content.shape == mask.shape:
                        return pred(content, mask).tolist()
    return None

def _case_overlay_objects(pairs, ti):
    # RE-ARC "OverlayObjectsGenerator": input = vconcat(g1, g2) (g1 top half,
    # g2 bottom half, equal heights); output = g1 with g2's non-bg cells
    # painted over it.
    if any(i.shape[0] % 2 for i, o in pairs):
        return None
    halves = []
    for i, o in pairs:
        h = i.shape[0]
        top, bottom = i[:h // 2], i[h // 2:]
        if top.shape != o.shape or top.shape != bottom.shape:
            return None
        halves.append((top, bottom, o))
    for bg in _bg_candidates(pairs):
        def pred(top, bottom, bg=bg):
            return np.where(bottom != bg, bottom, top)
        if all(np.array_equal(pred(t, b), o) for t, b, o in halves):
            h = ti.shape[0]
            if h % 2 == 0:
                top, bottom = ti[:h // 2], ti[h // 2:]
                if top.shape == bottom.shape:
                    return pred(top, bottom).tolist()
    return None

def _case_connect_dots(pairs, ti):
    # RE-ARC "ConnectDotsGenerator": exactly two same-colored non-bg cells,
    # sharing a row or column -- fill the straight segment between them with
    # that color. Degenerate inputs (0, 1, or >2 non-bg cells, or two cells
    # not sharing a row/col) leave the grid unchanged -- this matches the
    # real generator's own degenerate cases (verified: 800/800 real train+
    # test pairs across 200 sampled tasks reproduced exactly, including the
    # rare case where the two random dot positions happen to collide into
    # one cell).
    if any(i.shape != o.shape for i, o in pairs):
        return None
    for bg in _bg_candidates(pairs):
        def connect(a, bg=bg):
            ys, xs = np.nonzero(a != bg)
            out = a.copy()
            if len(ys) != 2:
                return out
            color = a[ys[0], xs[0]]
            if a[ys[1], xs[1]] != color:
                return out
            if ys[0] == ys[1]:
                r = ys[0]
                c1, c2 = sorted([xs[0], xs[1]])
                out[r, c1:c2 + 1] = color
            elif xs[0] == xs[1]:
                c = xs[0]
                r1, r2 = sorted([ys[0], ys[1]])
                out[r1:r2 + 1, c] = color
            return out
        if all(np.array_equal(connect(i), o) for i, o in pairs):
            return connect(ti).tolist()
    return None

def _case_tile_and_color(pairs, ti):
    # RE-ARC "TileAndColorGenerator": tile the input by a constant ratio
    # (like case 3) THEN apply a consistent global color remap (like case 5,
    # but composed with tiling instead of a geometric op).
    ratios = set()
    ok = True
    for i, o in pairs:
        if i.shape[0] == 0 or i.shape[1] == 0 or o.shape[0] % i.shape[0] or o.shape[1] % i.shape[1]:
            ok = False
            break
        ratios.add((o.shape[0] // i.shape[0], o.shape[1] // i.shape[1]))
    if not ok or len(ratios) != 1:
        return None
    fr, fc = ratios.pop()
    if fr * fc <= 1:
        return None
    for fn in [lambda g: np.tile(g, (fr, fc)),
               lambda g: np.repeat(np.repeat(g, fr, axis=0), fc, axis=1)]:
        tp = [(fn(i), o) for i, o in pairs]
        if any(gt.shape != o.shape for gt, o in tp):
            continue
        cm = _vt_colormap(tp)
        if cm is None:
            continue
        if all(np.array_equal(np.vectorize(lambda x: cm.get(x, x))(gt), o) for gt, o in tp):
            gt_ti = fn(ti)
            return np.vectorize(lambda x: cm.get(x, x))(gt_ti).tolist()
    return None


_VT_CASES_CORE = [
    _case_geom, _case_colormap, _case_tile_upscale, _case_constant,
    _case_geom_then_colormap, _case_downscale,
]
_VT_CASES_EXTENDED = [
    # _case_border before _case_crop_to_content: border-removal is a MORE
    # SPECIFIC rule (strip exactly a fixed 1-cell ring, any color) than
    # crop-to-content (crop to a variable-width same-background margin) --
    # v34 fix, found by validating against RemoveBorderGenerator: when a
    # task's few sampled train pairs all happen to use a border color close
    # to what crop-to-content infers as "background", crop-to-content can
    # silently claim the match first and generalize wrong on a test pair
    # whose border color falls outside that inferred background. Trying the
    # more constrained rule first removes that ambiguity.
    _case_border, _case_crop_to_content, _case_gravity, _case_symmetry_repair,
    _case_fill_enclosed, _case_majority_color, _case_diagonal,
    _case_largest_object, _case_xor_halves, _case_unique_color,
    _case_count_objects, _case_apply_mask, _case_overlay_objects,
    _case_connect_dots, _case_tile_and_color,
]

def _vt_case_list():
    if os.getenv("ARC_VT_EXTENDED_CASES", "1") == "1":
        # v34 fix: _case_constant is a broad, low-specificity fallback -- with
        # only a handful of train pairs it can "verify" against outputs that
        # are constant only by coincidence (found via real-generator
        # validation: RE-ARC's CountObjectsGenerator and
        # DrawDiagonalGenerator both occasionally sample the same count/grid
        # for every train pair), preempting a more specific extended case
        # that would have generalized correctly to the test input. Moving it
        # to run LAST -- after every more specific extended case has had a
        # chance -- fixes this without changing what it can ever match.
        # Gated the same as every other v34 case: flag=0 keeps
        # _VT_CASES_CORE's original v15/v17 order exactly.
        core_no_constant = [c for c in _VT_CASES_CORE if c is not _case_constant]
        return core_no_constant + _VT_CASES_EXTENDED + [_case_constant]
    return _VT_CASES_CORE

def verified_transform_solve(train, test_input):
    try:
        pairs = [(np.array(p["input"]), np.array(p["output"])) for p in train]
        ti = np.array(test_input)
    except Exception:
        return None
    if not pairs:
        return None
    for case_fn in _vt_case_list():
        try:
            result = case_fn(pairs, ti)
        except Exception:
            result = None
        if result is not None:
            return result
    return None


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)

    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def _fallback_attempts(self, k, i):
        # v15: smart fallback for test keys the LLM decoded ZERO candidates for
        # (was blind [[0]] before). fill_submission() always overwrites this
        # when real decoder results exist, so this can only help, never hurt.
        test_in = self.queries[k]['test'][i]['input']
        train = self.queries[k]['train']
        try:
            sol = verified_transform_solve(train, test_in)
        except Exception:
            sol = None
        if sol is not None:
            a1 = sol
            a2 = test_in                                  # diverse 2nd guess
        else:
            a1 = test_in                                  # ~67% of tasks are same-shape
            a2 = np.array(test_in)[::-1, :].tolist()      # flipped: cheap, distinct
        if a2 == a1:
            a2 = np.array(test_in)[:, ::-1].tolist()
        return a1, a2

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {}
        for k in self.keys:
            outs = []
            for i in range(len(self.queries[k]['test'])):
                a1, a2 = self._fallback_attempts(k, i)
                outs.append({'attempt_1': a1, 'attempt_2': a2})
            submission[k] = outs
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]

# ---- v15: reciprocal-rank-fusion ensemble of the two selection algorithms ----
# Pure CPU post-processing over already-decoded candidates -- zero GPU cost,
# zero added wall-clock during the 12h decode loop. Cannot make a task worse
# that either algorithm alone already solves correctly at rank 0 (RRF keeps
# strong-consensus picks on top); it can only rescue tasks where the two
# algorithms disagree and the true answer is a close runner-up in both.
def rank_fuse(decoded_results, algos=None, k_const=60):
    algos = algos or selection_algorithms
    fused = {}
    for bk, guesses in decoded_results.items():
        rank_maps = []
        sol_by_hash = {}
        for algo in algos:
            ordered = algo(guesses)
            rmap = {}
            for idx, sol in enumerate(ordered):
                h = hashable(sol)
                rmap[h] = idx
                sol_by_hash[h] = sol
            rank_maps.append(rmap)
        scored = []
        for h in sol_by_hash:
            score = sum(1.0 / (k_const + rmap.get(h, 10**6)) for rmap in rank_maps)
            scored.append((score, sol_by_hash[h]))
        scored.sort(key=lambda x: x[0], reverse=True)
        fused[bk] = [o for _, o in scored]
    return fused


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def run_rank_fusion(self, algos=None, k_const=60):
        return rank_fuse(self.decoded_results, algos=algos, k_const=k_const)

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

        # v15: also report the RRF-fused ranking, so its win/loss vs. score_kgmon
        # (the notebook's chosen selection algo) is visible before trusting it.
        fused = self.run_rank_fusion()
        correct_puzzles = {k for k, v in fused.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
        print(correct_puzzles)
        score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
        print(f" acc: {score:5.1f}/{num_puzzles:3} ('rank_fuse [score_full_probmul_3, score_kgmon]')")


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]

    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()

        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):

            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()

        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():

            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")

        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    worker(rank, queue, end_time)

    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
import json, os, random, time

# v34: this is the mechanism that makes the RE-ARC dataset a REAL, measured,
# runtime dependency of the submission -- not just a "does the file exist"
# diagnostic print. It runs the CPU verified-transform fallback (the exact
# same arc_loader.verified_transform_solve used later by starter.py's
# workers and by cell7_submission.py's vote injection) against a real,
# time-boxed sample of the mounted RE-ARC dataset's own tasks, and uses the
# MEASURED wrong-answer rate to decide -- before the 12h GPU run starts --
# whether ARC_VT_EXTENDED_CASES is safe to leave on for this run. If the
# live number looks bad, it downgrades automatically to the core v15/v17
# cases only. No GPU, no training, no change to worker()/turbo_dfs/
# starter.py's dispatch or timing -- this cell finishes and exits well
# before the run cell even starts.
#
# Local validation (offline, against the real re_arc_generators.py source,
# 30 samples x 35 categories = 900 tasks) measured 0.22% wrong-answer rate
# after the v34 case fixes below the SAFE_WRONG_RATE threshold picked here
# -- this cell re-measures that number LIVE, against the actual mounted
# dataset's content, so the safety decision reflects real data rather than
# a number baked into this notebook at build time.

REARC_VALIDATE_BUDGET_SEC = int(os.getenv("ARC_REARC_VALIDATE_BUDGET_SEC", "720"))  # ~12min default
SAFE_WRONG_RATE = float(os.getenv("ARC_REARC_SAFE_WRONG_RATE", "0.02"))  # 2%

_REARC_CANDIDATES = [
    "/kaggle/input/datasets/dineshkumar0705/re-arc-dataset/dataset_1_re_arc/re_arc_dataset.json",
    "/kaggle/input/re-arc-dataset/dataset_1_re_arc/re_arc_dataset.json",
    "/kaggle/input/re-arc-dataset/re_arc_dataset.json",
]


def _rearc_load_tasks():
    path = next((p for p in _REARC_CANDIDATES if os.path.isfile(p)), None)
    if path is None:
        print(f"=== RE-ARC live validation === dataset not found at any of {_REARC_CANDIDATES} "
              f"-- skipping; ARC_VT_EXTENDED_CASES stays at its configured value "
              f"('{os.getenv('ARC_VT_EXTENDED_CASES', '1')}').")
        return None
    try:
        with open(path) as f:
            raw = json.load(f)
    except Exception as e:
        print(f"=== RE-ARC live validation === found {path} but failed to parse it as JSON ({e}) "
              f"-- skipping; ARC_VT_EXTENDED_CASES stays at its configured value.")
        return None
    items = list(raw.values()) if isinstance(raw, dict) else list(raw)
    print(f"=== RE-ARC live validation === loaded {len(items)} tasks from {path}")
    return items


def _rearc_validate_and_gate():
    from arc_loader import verified_transform_solve  # local import: file only exists on disk by this point

    items = _rearc_load_tasks()
    if not items:
        return

    random.Random(0).shuffle(items)
    n_ok = n_wrong = n_none = n_total = n_skipped = 0
    t0 = time.time()
    last_print = t0
    for task in items:
        if time.time() - t0 > REARC_VALIDATE_BUDGET_SEC:
            break
        try:
            train, test = task["train"], task["test"]
            if not train or not test:
                n_skipped += 1
                continue
            test_pair = test[0]
        except Exception:
            n_skipped += 1
            continue
        try:
            pred = verified_transform_solve(train, test_pair["input"])
        except Exception:
            pred = None
        n_total += 1
        if pred is None:
            n_none += 1
        elif pred == test_pair["output"]:
            n_ok += 1
        else:
            n_wrong += 1
        now = time.time()
        if now - last_print > 60:
            print(f"  ...{n_total} checked so far ({now - t0:.0f}s elapsed, "
                  f"{n_ok} solved / {n_wrong} wrong / {n_none} declined)")
            last_print = now

    elapsed = time.time() - t0
    wrong_rate = n_wrong / max(n_total, 1)
    solve_rate = n_ok / max(n_total, 1)
    print(f"=== RE-ARC live validation === {n_total} real dataset tasks checked in {elapsed:.0f}s "
          f"({n_skipped} skipped as malformed) -> {n_ok} solved / {n_wrong} wrong / {n_none} declined "
          f"= {100*solve_rate:.1f}% solve rate, {100*wrong_rate:.2f}% wrong-answer rate")

    if wrong_rate > SAFE_WRONG_RATE and os.getenv("ARC_VT_EXTENDED_CASES", "1") == "1":
        os.environ["ARC_VT_EXTENDED_CASES"] = "0"
        print(f"=== RE-ARC live validation === measured wrong-answer rate {100*wrong_rate:.2f}% exceeds "
              f"the {100*SAFE_WRONG_RATE:.0f}% safety threshold -- auto-disabling ARC_VT_EXTENDED_CASES "
              f"for the rest of this run (falls back to the 6 core v15/v17 fallback cases only). This is "
              f"a real decision driven by the mounted dataset's actual content, made before the 12h GPU "
              f"run starts -- starter.py inherits this via its subprocess environment.")
    else:
        print(f"=== RE-ARC live validation === within the {100*SAFE_WRONG_RATE:.0f}% safety threshold -- "
              f"ARC_VT_EXTENDED_CASES stays at '{os.getenv('ARC_VT_EXTENDED_CASES', '1')}' for this run.")


_rearc_cell_start = time.time()
try:
    _rearc_validate_and_gate()
except Exception as e:
    print(f"=== RE-ARC live validation === skipped due to an unexpected error ({e}); "
          f"ARC_VT_EXTENDED_CASES stays at its configured value. This cell can never block or fail "
          f"the run -- it only ever reads and optionally tightens one env var.")

# v34: push global_end_time out by however long this cell actually took, so
# the GPU/TTT/decode run below still gets its full, undiminished 12h budget
# -- this cell's own runtime is pure ADDED overhead (the extra ~10-20min),
# never time borrowed from the GPU run, keeping the "same GPU/TTT/decode
# timings as the v15/v17 baseline" claim above literally true regardless of
# how long dataset loading/validation actually takes.
try:
    _rearc_cell_elapsed = time.time() - _rearc_cell_start
    global_end_time += _rearc_cell_elapsed
    print(f"=== RE-ARC live validation === this cell took {_rearc_cell_elapsed:.0f}s; "
          f"extended global_end_time by the same amount so the GPU run's own 12h budget is unchanged.")
except NameError:
    pass  # global_end_time not defined yet (diagnostic cell didn't run) -- nothing to extend


In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
import numpy as np
from arc_loader import ArcDataset, verified_transform_solve, is_valid_solution
from arc_decoder import ArcDecoder, score_kgmon

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

# v34: OPTIONAL -- inject the CPU verified-transform candidate into the real
# vote, not just the zero-candidate default (ARC_INCLUDE_VT_IN_VOTE, off by
# default). Different from the always-on fallback in arc_loader.py's
# get_submission(): that one only ever fires for keys the decoder produced
# ZERO candidates for. This flag additionally lets a verified (i.e. exactly
# reproduces every train pair) candidate compete for keys the decoder DID
# produce guesses for, by adding it as one more guess into the same
# rank_fuse/score_kgmon vote pool below -- so it can only change the outcome
# where the LLM's own candidates were weak/scattered, not where they already
# agree strongly (score_sum groups guesses by identical solution and rewards
# groups with more supporting members, so a lone synthetic candidate can't
# outvote a well-supported consensus).
#
# The injected candidate's beam_score/score_aug are set to JUST BETTER (a
# fixed 1e-6 nudge) than the BEST score already present among the real
# guesses in that key's bucket -- not an artificially large confidence
# invented from nothing. Two designs were tried and empirically rejected
# before this one (see test_vt_in_vote.py): a median-of-bucket score is too
# weak to ever beat even a single existing guess (near no-op, never changes
# the top pick); an exact tie at the bucket's best score LOSES the tie under
# score_sum's stable sort (existing guesses were inserted first), which is
# just as inert. A tiny nudge below the current best breaks the tie in the
# verified candidate's favor -- defensible because "reproduces 100% of the
# train pairs exactly" really is at least as strong a signal as any single
# beam sample's log-probability -- while staying negligible next to real
# score gaps (all observed beam scores in this notebook's own benchmark
# output are >=1e-3 apart), so it can't manufacture false confidence, and it
# still can't outrank a well-supported multi-guess LLM consensus: score_sum
# rewards groups by how many guesses support them, not just the best
# individual score, so a lone synthetic candidate loses to any group with
# more than one real supporter (verified both directions in
# test_vt_in_vote.py). Pure CPU, runs once over already-decoded results
# after the 12h GPU loop -- zero effect on GPU/training timing either way.
#
# UNTESTED against a full eval run (unlike arc_loader.py's cases 1-6, this
# specific mechanism has no held-out-benchmark number behind it yet) -- keep
# this OFF until you've A/B'd it, same as this codebase's other experimental
# flags (ARC_USE_RANK_FUSION, ARC_LORA_RANK, ...).
use_vt_in_vote = os.getenv("ARC_INCLUDE_VT_IN_VOTE", "0") == "1"
if use_vt_in_vote:
    n_injected = 0
    for bk, guesses in decoder.decoded_results.items():
        q = decoder.dataset.queries.get(bk)
        if not q or not q.get("test"):
            continue
        test_in = q["test"][0]["input"]
        try:
            vt_sol = verified_transform_solve(q["train"], test_in)
        except Exception:
            vt_sol = None
        if vt_sol is None:
            continue
        vt_arr = np.array(vt_sol)
        if not is_valid_solution(vt_arr):
            continue
        beam_scores = [g["beam_score"] for g in guesses.values()]
        if not beam_scores:
            continue
        _eps = 1e-6
        aug_all = np.concatenate([np.asarray(g["score_aug"]) for g in guesses.values()])
        aug_lens = [len(g["score_aug"]) for g in guesses.values()]
        guesses[f"{bk}.vts_synthetic"] = {
            "beam_score": float(np.min(beam_scores)) - _eps,
            "score_aug": [float(np.min(aug_all)) - _eps] * int(np.median(aug_lens)),
            "solution": vt_arr,
        }
        n_injected += 1
    print(f"*** ARC_INCLUDE_VT_IN_VOTE=1: injected verified-transform candidates into "
          f"{n_injected}/{len(decoder.decoded_results)} vote buckets")

# v15: reciprocal-rank-fusion ensemble of both selection algorithms, zero GPU
# cost (pure CPU post-processing over already-decoded candidates). Toggle off
# with ARC_USE_RANK_FUSION=0 to fall back to the proven v10 (30.42) behavior
# (score_kgmon alone) -- one env var, no code change, per the usual A/B doctrine.
use_rank_fusion = os.getenv("ARC_USE_RANK_FUSION", "1") == "1"
if use_rank_fusion:
    results = decoder.run_rank_fusion()
else:
    results = decoder.run_selection_algo(score_kgmon)

# get_submission() now fills any test key with ZERO decoded candidates using a
# verified-transform CPU solver instead of a blind [[0]] guess (see arc_loader.py).
# fill_submission() below always overwrites keys that DO have real results, so
# this cannot regress any task the decoder actually solved.
submission = data.get_submission(results)

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))
    print(f"*** rank fusion {'ENABLED' if use_rank_fusion else 'disabled'} for this submission (ARC_USE_RANK_FUSION)")
    print(f"*** VT-in-vote {'ENABLED' if use_vt_in_vote else 'disabled'} for this submission (ARC_INCLUDE_VT_IN_VOTE)")
